In [23]:
# goal: prototype skeleton transformer
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from map4 import MAP4
import re

from pathlib import Path

In [24]:
import numpy as np
import torch

class RadialSeeker:

    def __init__(self, radial_resolution, intrashell_resolution, max_angstroms,
                 verbose=False):
        self.radial_resolution = radial_resolution
        self.angstrom_lim = max_angstroms   # maximum angstrom range found + some extra for context enrichment
                                 # can always edit this later on
        self.angstrom_inc = float(max_angstroms / radial_resolution)
        self.threshold = self.angstrom_inc / 2 # standardize how we determine radial sequences
        # getting half the angstrom increment will let us finely separate them

        #                           smallest distance is the base increment, not 0
        self.radius_levels = torch.arange(self.angstrom_lim, self.angstrom_inc, step= -1 * self.angstrom_inc)

        self.intrashell_resolution = intrashell_resolution
        self.intrashell_inc = max_angstroms / intrashell_resolution

        self.verbose = verbose


    def radial_sequence(self, aa_seq, vect_seq):
        # first order them by absolute distance to centroid
        # then convert them to indices according to intra-shell resolution

        # radial resolution determines how finely we want to order them
        # shell resolution determines how finely in 3D space we represent them

        radial_seq = []  # lookup table
        seen = set()

        # sanity
        if len(aa_seq) != len(vect_seq):
            raise ValueError(f"string and vector sequences of {aa_seq} are different lengths")

        # iterate up resolution increments, if a molecule's coordinate is within like 1/resolution of an angstrom, append it's info
        # its (radius, pos index) is now the unique ID, so if we stumble on it again its not duplicated
        for level in self.radius_levels: # go through all possible radius levels
            for i in range(len(aa_seq)):  # go through all amino acids in that sequence
                num_id=i
                if num_id not in seen and self._dist_check(np.linalg.norm(vect_seq[i]), level):
                    idx_vect = self.vect2idx(vect_seq[i])
                    radial_seq.append([[aa_seq[i]], idx_vect])
                    seen.add(num_id)

        # create two VOID tokens (0,0,0) on both sides to denote stops and starts
        return [[['VOID'],[0, 0, 0]]] + radial_seq + [[['VOID'],[0, 0, 0]]]  # we want to go from outside inward

    def _dist_check(self, dist, ang_radius):
        if abs(dist - ang_radius) <= self.threshold:
            return True
        else:
            return False

    def vect2idx(self, vector):
        """
        Converts a torch.tensor of 3D vectors into their nearest shell-resolution's index
        """
        idxs = np.round(vector / self.intrashell_inc)
        idxs = np.clip(idxs, -self.intrashell_resolution, self.intrashell_resolution)
        return idxs.astype(int)

    def idx2vect(self, idxs):
        """Converts a torch.tensor of indexes back into their original 3D vectors"""
        return idxs * self.intrashell_inc


def test_resolution():
    module = RadialSeeker(radial_resolution = 100, intrashell_resolution = 100, max_angstroms = 33)
    for level in module.radius_levels:
        print(level)
# test_resolution()  # all good

In [25]:
mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

# will be important later for visualization of skeleton
def eval_lipinski(smiles):
    """
    Use on user-input smiles, don't shut down inference run just flag molecule as non drug-like
    :param smiles:
    :return:
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES String")

    try:
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        h_donors = Lipinski.NumHDonors(mol)
        h_acceptors = Lipinski.NumHAcceptors(mol)
    except Exception as e:
        raise ValueError(f"Error calculating Lipinski Descriptors: {e}")

    rules_passed = 0
    if mw <= 500: rules_passed += 1
    if logp <= 5: rules_passed += 1
    if h_donors <= 5: rules_passed += 1
    if h_acceptors <= 10: rules_passed += 1

    return rules_passed

def mol_from_smiles(smiles):
    """
    Extract molecular fingerprint from singular SMILES - fairly straightforward
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None

In [41]:
# Load data
data_folder = Path("notebook_database")
RADIAL_SEQUENCES = pd.read_csv(data_folder/"radial_seq_df.csv")
MOLECULAR_FINGERPRINTS = pd.read_csv(data_folder/"molfp_df.csv")
SMILE_2_PDBHITS = pd.read_csv(data_folder/"SMILE_2_PDBhits.csv")

import ast
import re
def clean_radial_seq(s):
    """Pandas can be annoying"""
    s = re.sub(r'array\(([\[\]0-9,\s\-]+)\)', lambda m: m.group(1).replace('\n', ''), s)
    return ast.literal_eval(s)

RADIAL_SEQUENCES["radial_sequence"] = RADIAL_SEQUENCES["radial_sequence"].apply(clean_radial_seq)

In [43]:
RADIAL_SEQUENCES["radial_sequence"].head(10)

0    [[[VOID], [0, 0, 0]], [[Y], [-30, -24, -27]], ...
1    [[[VOID], [0, 0, 0]], [[I], [-22, 14, -34]], [...
2    [[[VOID], [0, 0, 0]], [[I], [34, -11, -20]], [...
3    [[[VOID], [0, 0, 0]], [[E], [-18, -39, 20]], [...
4    [[[VOID], [0, 0, 0]], [[Y], [21, 9, -32]], [[L...
5    [[[VOID], [0, 0, 0]], [[E], [-30, -22, -5]], [...
6    [[[VOID], [0, 0, 0]], [[I], [31, -15, 27]], [[...
7    [[[VOID], [0, 0, 0]], [[K], [-37, 27, -1]], [[...
8    [[[VOID], [0, 0, 0]], [[L], [-6, -44, -3]], [[...
9    [[[VOID], [0, 0, 0]], [[W], [20, -1, -46]], [[...
Name: radial_sequence, dtype: object

In [30]:
batch_size = 64
block_size = 20
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 3
dropout = 0.2

torch.manual_seed(311104)

In [53]:
# need to think of how we're about to load up our data
# i guess a nested for loop is how we get every combination the easiest?
# print(SMILE_2_PDBHITS.columns)  # 'SMILES', 'PDB_hits'
# print(MOLECULAR_FINGERPRINTS.columns)  # 'smiles_str', 'map4_fp'
# print(RADIAL_SEQUENCES.columns)  # 'PDB_ID', 'radial_sequence'

test_radial_seq = RADIAL_SEQUENCES.iloc[0]["radial_sequence"]

def radial_XY(radial_seq, block_size):
    """
    Generates X and Y set for a given radial sequence
    :param radial_seq:
    :param block_size:
    :return:
    """
    X, Y = [], []
    context = [0,0,0] * block_size
    for idx in radial_seq:
        coords = idx[1]
        X.append(context)
        Y.append(coords)
        context = context[3:] + coords

    return X, Y

X, Y = radial_XY(radial_seq=test_radial_seq, block_size=10)
for pos in range(len(X)):
    print(f"{X[pos]} => {Y[pos]}")

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] => [0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] => [-30, -24, -27]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27] => [11, -33, -11]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27, 11, -33, -11] => [20, -26, -14]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27, 11, -33, -11, 20, -26, -14] => [27, -14, 17]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27, 11, -33, -11, 20, -26, -14, 27, -14, 17] => [-28, -17, 10]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27, 11, -33, -11, 20, -26, -14, 27, -14, 17, -28, -17, 10] => [5, 23, 25]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -30, -24, -27, 11, -33, -11, 20, -26, -14, 27, -14, 17, -28, -17, 10, 5, 23, 25] => [-8, -33, 7]
[0, 0, 0, 0, 0, 0, 0, 0, 0, -30, 

In [ ]:
# Generate X and Y total